In [1]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

Working directory: C:\Users\makar\Python\auto-data-scientist


In [2]:
%%writefile src/llm.py
import os
import time

from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIConnectionError, APITimeoutError

load_dotenv()   # reads LLM_BASE_URL, LLM_API_KEY, LLM_MODEL from your .env file

_client = None  # created on first use, then reused


def get_client() -> OpenAI:
    """One OpenAI-style client works for Gemini, Groq, Cerebras, Mistral, Ollama..."""
    global _client
    if _client is None:
        _client = OpenAI(
            base_url=os.environ["LLM_BASE_URL"],
            api_key=os.environ["LLM_API_KEY"],
            timeout=120,          # give up on a single call after 2 minutes
        )
    return _client


def chat(messages: list, temperature: float = 0.2, max_retries: int = 5) -> str:
    """Send a list of chat messages to the LLM and return its text reply.

    Retries with growing waits when we hit rate limits or network hiccups.
    Other errors (wrong key, wrong model name) are raised immediately,
    because retrying would not help.
    """
    client = get_client()
    model = os.environ["LLM_MODEL"]

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
            )
            return resp.choices[0].message.content or ""

        except (RateLimitError, APIConnectionError, APITimeoutError) as e:
            if attempt == max_retries - 1:
                raise                                  # out of retries: give up
            wait = min(60, 5 * 2 ** attempt)           # 5s, 10s, 20s, 40s, 60s
            print(f"[llm] {type(e).__name__}: retrying in {wait}s "
                  f"(attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)

Overwriting src/llm.py


In [3]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
from src.llm import chat

print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))
# EXPECTED: "ready"

ready


In [5]:
pip install --upgrade brotli

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install --upgrade brotlicffi

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Which model IDs does your account offer? (IDs must match .env exactly)
from src.llm import get_client
ids = sorted(m.id for m in get_client().models.list())
print([i for i in ids if i.startswith("gpt")])

[]


In [8]:
from src.llm import get_client
ids = sorted(m.id for m in get_client().models.list())
print(ids)

['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


In [9]:
from src.llm import chat
print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))

ready


In [10]:
%%writefile src/profiler.py
import pandas as pd


def quick_profile(csv_path, max_cols: int = 60) -> str:
    """Return a short text summary of a CSV that an LLM can read.

    We send this instead of the raw data: it is small, cheap, and does not
    leak the actual rows.
    """
    df = pd.read_csv(csv_path)

    lines = [
        f"Rows: {len(df)}, Columns: {df.shape[1]}",
        "",
        "Columns (name | dtype | missing values | unique values | example):",
    ]
    for col in df.columns[:max_cols]:
        s = df[col]
        example = s.dropna().iloc[0] if s.notna().any() else None
        lines.append(f"- {col} | {s.dtype} | {s.isna().sum()} | {s.nunique()} | {example}")

    if df.shape[1] > max_cols:
        lines.append(f"... and {df.shape[1] - max_cols} more columns not shown")

    return "\n".join(lines)

Writing src/profiler.py


In [13]:
from pathlib import Path
Path("src/agents").mkdir(parents=True, exist_ok=True)

In [14]:
%%writefile src/agents/__init__.py
# Marks src/agents/ as a package

Writing src/agents/__init__.py


In [15]:
%%writefile src/agents/coder.py
import re
from dataclasses import dataclass, field
from pathlib import Path

from src.llm import chat
from src.sandbox import run_in_sandbox, SandboxResult

# Three backticks, built this way so they can't be mistaken for Markdown formatting
FENCE = "`" * 3

# The "system" message: standing rules the LLM must follow on every call
SYSTEM_PROMPT = f"""You are an expert data scientist. You write Python code that is executed automatically in a sandbox.

ENVIRONMENT RULES
- There is no internet access. Never try to download anything.
- The dataset is the file 'data.csv' in the current directory.
- Available libraries: pandas (3.x), numpy, scikit-learn, xgboost, matplotlib, seaborn, joblib.
- Only what you print() is returned to you, so print key results (metrics, shapes, etc.).
- For charts: call matplotlib.use("Agg") before importing pyplot, save with plt.savefig("name.png"), and never call plt.show().
- Save any files (charts, models) in the current directory.
- pandas 3.x: never use chained assignment or inplace=True on a column. Write df["col"] = df["col"].fillna(value).

OUTPUT FORMAT
- Reply with exactly ONE complete Python script inside a single code block that starts with {FENCE}python and ends with {FENCE}.
- Do not write explanations outside the code block. Use short comments inside the code instead.
"""

# Pattern that finds code blocks in the LLM's reply
CODE_BLOCK = re.compile(FENCE + r"(?:python|py)?[ \t]*\n(.*?)" + FENCE, re.DOTALL)


def extract_code(reply: str) -> str:
    """Pull the Python code out of the LLM's reply.

    LLMs wrap code in a fenced block and sometimes add chatty text around it.
    If there are several blocks we take the longest one (the real script).
    If there are none, we assume the whole reply is code.
    """
    blocks = CODE_BLOCK.findall(reply)
    if blocks:
        return max(blocks, key=len).strip()
    return reply.strip()


@dataclass
class StepOutcome:
    """What the agent hands back after working on one task."""
    success: bool
    attempts: int
    code: str
    result: SandboxResult
    history: list = field(default_factory=list)   # every attempt, for debugging


def solve_step(task: str, data_profile: str, workspace: Path,
               max_attempts: int = 4, verbose: bool = True) -> StepOutcome:
    """Write code for `task`, run it in the sandbox, and repair it if it fails."""
    base = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"TASK:\n{task}\n\nDATA PROFILE (file: data.csv):\n{data_profile}"},
    ]
    messages = list(base)
    history = []

    for attempt in range(1, max_attempts + 1):
        reply = chat(messages)                 # 1) ask the LLM for code
        code = extract_code(reply)

        if code:                               # 2) run it safely
            result = run_in_sandbox(code, workspace)
        else:                                  # the model returned nothing usable
            result = SandboxResult(ok=False, stdout="", stderr="Your reply contained no code.",
                                   returncode=-2, timed_out=False)

        history.append({"attempt": attempt, "code": code, "summary": result.summary()})
        if verbose:
            print(f"[coder] attempt {attempt}/{max_attempts}: {'OK' if result.ok else 'FAILED'}")

        if result.ok:                          # 3) success: stop here
            return StepOutcome(True, attempt, code, result, history)

        # 4) failure: rebuild the conversation as
        #    original request + its last answer + the error.
        #    We do NOT keep every past attempt, which saves tokens on free tiers.
        messages = base + [
            {"role": "assistant", "content": reply},
            {"role": "user", "content": (
                "Your script failed when executed.\n\n"
                f"{result.summary()}\n\n"
                "Read the error carefully, fix the root cause, and reply with the "
                "full corrected script in one code block."
            )},
        ]

    return StepOutcome(False, max_attempts, code, result, history)

Writing src/agents/coder.py


In [16]:
from sklearn.datasets import load_breast_cancer
from src.config import RUNS_DIR
from src.profiler import quick_profile

ws = RUNS_DIR / "part2_test" / "workspace"
ws.mkdir(parents=True, exist_ok=True)

# A built-in medical dataset, saved as data.csv. Its target column is named "target".
load_breast_cancer(as_frame=True).frame.to_csv(ws / "data.csv", index=False)

profile = quick_profile(ws / "data.csv")
print(profile)

Rows: 569, Columns: 31

Columns (name | dtype | missing values | unique values | example):
- mean radius | float64 | 0 | 456 | 17.99
- mean texture | float64 | 0 | 479 | 10.38
- mean perimeter | float64 | 0 | 522 | 122.8
- mean area | float64 | 0 | 539 | 1001.0
- mean smoothness | float64 | 0 | 474 | 0.1184
- mean compactness | float64 | 0 | 537 | 0.2776
- mean concavity | float64 | 0 | 537 | 0.3001
- mean concave points | float64 | 0 | 542 | 0.1471
- mean symmetry | float64 | 0 | 432 | 0.2419
- mean fractal dimension | float64 | 0 | 499 | 0.07871
- radius error | float64 | 0 | 540 | 1.095
- texture error | float64 | 0 | 519 | 0.9053
- perimeter error | float64 | 0 | 533 | 8.589
- area error | float64 | 0 | 528 | 153.4
- smoothness error | float64 | 0 | 547 | 0.006399
- compactness error | float64 | 0 | 541 | 0.04904
- concavity error | float64 | 0 | 533 | 0.05373
- concave points error | float64 | 0 | 507 | 0.01587
- symmetry error | float64 | 0 | 498 | 0.03003
- fractal dimension err

In [17]:
from src.agents.coder import solve_step

task = ("Train a logistic regression model to predict the 'target' column. "
        "Use an 80/20 train/test split with a fixed random seed. "
        "Print the test accuracy and ROC-AUC, and save a confusion matrix chart as confusion.png.")

outcome = solve_step(task, profile, ws, max_attempts=4)

print("\nSuccess:", outcome.success, "| Attempts:", outcome.attempts)
print("\n--- STDOUT ---\n", outcome.result.stdout)
print("--- NEW FILES ---", outcome.result.new_files)

[coder] attempt 1/4: OK

Success: True | Attempts: 1

--- STDOUT ---
 Test Accuracy: 0.9825
Test ROC-AUC: 0.9957

--- NEW FILES --- ['confusion.png']


In [18]:
print(outcome.code)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("data.csv")

# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Train‑test split with fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression model
model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_tes